# 🟣 Gaussian Mixture Model (GMM)
**Module 1 — Clustering Algorithms**

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, load_iris
from sklearn.metrics import silhouette_score
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 2. What is GMM?
> A **probabilistic model** that assumes data is generated from a **mixture of K Gaussian distributions**.
Unlike K-Means (hard assignment), GMM gives **soft/probabilistic** cluster membership.

**GMM formula:**  `p(x) = Σ πₖ N(x | μₖ, Σₖ)`

| Parameter | Meaning |
|---|---|
| `πₖ` | Mixture weight (how likely cluster k is) |
| `μₖ` | Mean of Gaussian k |
| `Σₖ` | Covariance of Gaussian k |

**Covariance types:** `full`, `tied`, `diag`, `spherical`
Fitted via **EM Algorithm** (Expectation-Maximization)

## 3. Dataset

In [ ]:
X, y_true = make_blobs(n_samples=400, centers=3, cluster_std=[0.8, 1.2, 0.5], random_state=42)
print(f'Shape: {X.shape}')
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[:, 0], X[:, 1], c=y_true, cmap='Set1', s=30, alpha=0.7)
ax.set_title('Raw Data — 3 Clusters (varying spread)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Fit GMM and Visualize Gaussians

In [ ]:
def draw_ellipse(position, covariance, ax=None, **kwargs):
    ax = ax or plt.gca()
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width = height = 2 * np.sqrt(covariance)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height, angle=angle, **kwargs))

gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(X)
labels_gmm = gmm.predict(X)
probs = gmm.predict_proba(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = ['steelblue', 'coral', 'mediumseagreen']

# Soft probabilities
for i, color in enumerate(colors):
    axes[0].scatter(X[labels_gmm == i, 0], X[labels_gmm == i, 1],
                    c=color, s=30, alpha=0.6, label=f'Cluster {i}')
    draw_ellipse(gmm.means_[i], gmm.covariances_[i], ax=axes[0],
                 alpha=0.15, color=color)
axes[0].scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='black', s=200, marker='*', zorder=5, label='Means')
axes[0].set_title('GMM Clusters + Gaussian Ellipses', fontsize=13, fontweight='bold')
axes[0].legend()

# Probability scatter (max prob = confidence)
max_prob = probs.max(axis=1)
sc = axes[1].scatter(X[:, 0], X[:, 1], c=max_prob, cmap='RdYlGn', s=30, alpha=0.8, vmin=0.5, vmax=1)
plt.colorbar(sc, ax=axes[1], label='Confidence (max probability)')
axes[1].set_title('Soft Assignment Confidence', fontsize=13, fontweight='bold')

plt.suptitle('Gaussian Mixture Model', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. BIC / AIC — Selecting Number of Components

In [ ]:
n_components = range(1, 11)
bic_scores, aic_scores = [], []

for n in n_components:
    g = GaussianMixture(n_components=n, covariance_type='full', random_state=42, n_init=3)
    g.fit(X)
    bic_scores.append(g.bic(X))
    aic_scores.append(g.aic(X))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_components, bic_scores, 'bo-', lw=2, markersize=7, label='BIC')
ax.plot(n_components, aic_scores, 'rs-', lw=2, markersize=7, label='AIC')
ax.axvline(3, color='green', linestyle='--', lw=1.5, label='Optimal K=3')
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Score (lower = better)', fontsize=12)
ax.set_title('BIC / AIC for GMM Component Selection', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Best K by BIC: {np.argmin(bic_scores) + 1}')
print(f'Best K by AIC: {np.argmin(aic_scores) + 1}')

## 6. Covariance Type Comparison

In [ ]:
cov_types = ['full', 'tied', 'diag', 'spherical']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, cov in enumerate(cov_types):
    g = GaussianMixture(n_components=3, covariance_type=cov, random_state=42)
    g.fit(X)
    lbl = g.predict(X)
    sil = silhouette_score(X, lbl)
    bic = g.bic(X)

    colors = plt.cm.Set1(np.linspace(0, 0.8, 3))
    for j in range(3):
        axes[i].scatter(X[lbl == j, 0], X[lbl == j, 1], color=colors[j], s=20, alpha=0.6)
        cov_mat = g.covariances_[j] if cov in ['full', 'diag'] else (
                  g.covariances_ if cov == 'tied' else np.eye(2) * g.covariances_[j])
        if cov_mat.ndim < 2:
            cov_mat = np.eye(2) * g.covariances_[j]
        draw_ellipse(g.means_[j], cov_mat, ax=axes[i], alpha=0.12, color=colors[j])
    axes[i].scatter(g.means_[:, 0], g.means_[:, 1], c='black', s=150, marker='*', zorder=5)
    axes[i].set_title(f'cov_type="{cov}"\nSilhouette={sil:.3f} | BIC={bic:.1f}', fontsize=11, fontweight='bold')

plt.suptitle('GMM — Covariance Type Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Density Estimation — 2D Probability Landscape

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
XX = np.c_[xx.ravel(), yy.ravel()]
Z = np.exp(gmm.score_samples(XX)).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 8))
cf = ax.contourf(xx, yy, Z, levels=20, cmap='YlOrRd', alpha=0.8)
ax.scatter(X[:, 0], X[:, 1], c=labels_gmm, cmap='Set1', s=20, alpha=0.5, edgecolors='gray', lw=0.3)
plt.colorbar(cf, ax=ax, label='Density p(x)')
ax.set_title('GMM Probability Density Landscape', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. GMM vs K-Means — Soft vs Hard Assignment

In [ ]:
from sklearn.cluster import KMeans
X_test, y_test = make_blobs(n_samples=400, centers=3, cluster_std=[0.8, 1.5, 0.5], random_state=42)

km  = KMeans(n_clusters=3, random_state=42)
gm  = GaussianMixture(n_components=3, random_state=42)
lbl_km = km.fit_predict(X_test)
gm.fit(X_test)
lbl_gm = gm.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (lbl, title) in zip(axes, [(y_test, 'Ground Truth'), (lbl_km, 'K-Means (Hard)'), (lbl_gm, 'GMM (Soft→Hard)')]):
    ax.scatter(X_test[:, 0], X_test[:, 1], c=lbl, cmap='Set1', s=25, alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold')
plt.suptitle('K-Means vs GMM Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'K-Means Silhouette : {silhouette_score(X_test, lbl_km):.4f}')
print(f'GMM    Silhouette  : {silhouette_score(X_test, lbl_gm):.4f}')

## 9. Iris Dataset — GMM

In [ ]:
from sklearn.decomposition import PCA
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

gmm_iris = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm_iris.fit(X_iris)
lbl_iris = gmm_iris.predict(X_iris)

X_pca = PCA(n_components=2).fit_transform(X_iris)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lbl, title) in zip(axes, [(lbl_iris, 'GMM'), (iris.target, 'Ground Truth')]):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=lbl, cmap='Set1', s=40, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Silhouette Score (Iris): {silhouette_score(X_iris, lbl_iris):.4f}')
print(f'BIC: {gmm_iris.bic(X_iris):.2f}  |  AIC: {gmm_iris.aic(X_iris):.2f}')

## 10. Key Takeaways
> - GMM is a **probabilistic model** — gives soft cluster membership probabilities
> - Use **BIC/AIC** to select the number of components (not elbow method)
> - `covariance_type='full'` is most flexible; `'spherical'` is like probabilistic K-Means
> - Great for **density estimation** and datasets with **elliptical, overlapping** clusters
> - Sensitive to initialization — use `n_init > 1`